In [32]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [33]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [34]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [35]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [36]:
train_data.sample(5)

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
8870,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
6554,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
12900,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
2596,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."


In [37]:
train_data.shape

(14732, 3)

In [38]:
val_data.shape

(818, 3)

In [39]:
# random sampling

train_data = train_data.sample(n= 4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n= 500, random_state = 42).reset_index(drop = True)

In [40]:
train_data.shape

(4000, 3)

In [41]:
val_data.shape

(500, 3)

## Data Preprocessing

In [42]:

def clean_data(text):
    import re

    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)

    return text.strip()


In [43]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [44]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

## Tokenize Data

In [45]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [46]:

def tokenize(data):
    # T5 task prefix for summarization
    dialogue = "summarize: " + data["dialogue"]

    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        text_target=data["summary"],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    # Ignore padding positions when calculating training loss
    labels = targets["input_ids"]
    labels = [
        token_id if token_id != tokenizer.pad_token_id else -100
        for token_id in labels
    ]

    inputs["labels"] = labels
    return inputs


In [47]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [48]:
train_dataset[0]

{'input_ids': [21603, 10, 28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [49]:
# input ids - dialogue => token ids

# 1 = EOS, 0 => padding

# ATTENTION MASK
# 1 => valid token id
# 0 => padding values or invalid

#------------------------------------------
# LABELS - TARGET => summary token

In [50]:
len(train_dataset[0]["input_ids"])

512

In [51]:
type(train_dataset)
type(val_dataset)

list

## Working with our model

In [52]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [53]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## Training Arguments


In [55]:

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    learning_rate=3e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=0.05,
    save_total_limit=2,
    logging_steps=50,
    report_to="none"
)


In [56]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

# Train the model

In [57]:
# training the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.166752,1.926314
2,2.028971,1.877115
3,1.985947,1.850850
4,1.977863,1.836774
5,1.933193,1.833561
6,1.921132,1.830684


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=2.0424692662556967, metrics={'train_runtime': 1048.2535, 'train_samples_per_second': 22.895, 'train_steps_per_second': 2.862, 'total_flos': 3248203235328000.0, 'train_loss': 2.0424692662556967, 'epoch': 6.0})

In [58]:
#model load => fine-tune =>save the model

In [59]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [60]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core logic for summarization

In [61]:

def summarize_dialogue(dialogue):
    dialogue = "summarize: " + clean_data(dialogue)

    inputs = tokenizer(
        dialogue,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    model.to(device)
    model.eval()

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        targets = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=120,
            num_beams=4,
            length_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary


In [68]:
test_dialogue = """
John: Have you checked the website performance report?

Sarah: Yes. The website traffic increased by 25% this month, but the bounce rate is still high.

John: What is the current bounce rate?

Sarah: It's 48%, compared to 55% last month.

John: That's an improvement. What are we doing to reduce it further?

Sarah: We're redesigning the landing page and improving the page loading speed. The changes should be ready by Friday.

John: Great. Can you also prepare a report on mobile users?

Sarah: Sure. Mobile users account for 65% of our total traffic, and most of them visit the product pages.

John: Please include these findings in the marketing meeting presentation next Monday.

Sarah: No problem. I'll finish the report by Thursday so we have time to review it.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary:", summary)

Summary: John checked the website performance report. The current bounce rate is 48%, compared to 55% last month. Sarah is redesigning the landing page and improving the page loading speed. Sarah will finish the report by Thursday.


In [65]:

import torch
from tqdm.auto import tqdm

model.eval()

predictions = []
references = []

batch_size = 8

for start in tqdm(range(0, len(val_data), batch_size)):
    batch = val_data.iloc[start:start + batch_size]

    dialogues = [
        "summarize: " + clean_data(text)
        for text in batch["dialogue"].tolist()
    ]

    inputs = tokenizer(
        dialogues,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=120,
            num_beams=4
        )

    predictions.extend(
        tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )
    )

    references.extend(batch["summary"].tolist())

print("Predictions:", len(predictions))
print("References:", len(references))


  0%|          | 0/63 [00:00<?, ?it/s]

Predictions: 500
References: 500


In [64]:
!pip install bert-score

In [66]:

from rouge_score import rouge_scorer

# Initialize ROUGE evaluator
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

# Compare each prediction with its reference
scores = [
    scorer.score(reference, prediction)
    for reference, prediction in zip(references, predictions)
]

# Calculate average precision, recall, and F1
for metric in ["rouge1", "rouge2", "rougeL"]:
    precision = sum(s[metric].precision for s in scores) / len(scores)
    recall = sum(s[metric].recall for s in scores) / len(scores)
    f1 = sum(s[metric].fmeasure for s in scores) / len(scores)

    print(f"\n{metric.upper()}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")



ROUGE1
Precision: 0.4925
Recall:    0.4391
F1 Score:  0.4374

ROUGE2
Precision: 0.2308
Recall:    0.2028
F1 Score:  0.2017

ROUGEL
Precision: 0.4054
Recall:    0.3621
F1 Score:  0.3601
